# Cleaning4
Livello di pulizia che prende i file cleaned_01 fa le elaborazioni di cleaning 2 & 3 ma senza eliminare righe.\
File così creati servono per il merge.
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [2]:
file_codes = ['ADNI_DIAN_COMPARISON']

#'ADNIMERGE', 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES', 
#            'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS'

### Mixed info ###
# 'ADNIMERGE', 
# 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

### Single Cofactor ###
# 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

### Volumes ###
# 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS',
# 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'  --> just partial immages segmentation

### CSF ###
# 'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [3]:
search = client.query_files(
    query={'custom.level' : 'cleaned_01', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


In [4]:
len(zip_files)

1

## Operazioni
- Trasformare in dummies alcuni parametri
- Normalizzare i volumi
- nuovi metadati (cofattori e fattori)

In [5]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned1'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned4'

if os.path.isfile(new_name+'.xlsx'):
    #aggiunge i filecode mancanti e riporta i file_code da riprocessare allo status precedente (variable names)
    update_new_support_file(support_file, new_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)


The ADNI_variables_cleaned4 file has been updated with the new file_code: []
Open the file and verify it, if needed update the variables names and metadata
The ADNI_variables_cleaned4 file has restored the previous information of the file_code: ['ADNI_DIAN_COMPARISON']
Open the file and verify it, if needed update the variables names and metadata


In [6]:
new_support_file = pd.read_excel(new_name+'.xlsx')
dataCleaner = DataCleaner(support_file=new_support_file)

In [7]:
for file_name in zip_files.keys():
    print('\n\n ----', file_name)
    
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    new_support_file = dataCleaner.update_self_support_file(new_support_file)
    
    file_code, metadat_custom = dataCleaner.get_file_code_metadata(file_name, prefix='cleaned/single_file')

    processed_df = df_new.copy(deep=True)

    # funzione che trasforma parametri categorici in dummies
    ref_list = ['GENDER', 'MARRY', 'ETHNICITY', 'RACE', 'DX']
    to_dummy_list = [x for x in ref_list if x in processed_df.columns]
    if to_dummy_list:
        print('\n\n Dummies in ----', file_name)
        processed_df, bool_var = dataCleaner.classes_to_dummies(processed_df, col_list=to_dummy_list) 
    
    if 'volume' in support_file[support_file['file_code'] == file_code]['metadati_normalizzazione'].values:
        print('\n\n Volumes in ----', file_name)
        # verifica che i volumi siano già totali e non solo una parte laterale
        processed_df, volume_list = dataCleaner.get_volumes_total(processed_df, file_code)
        print(processed_df.columns)
        # Transform volumes as ICV percentage
        processed_df = dataCleaner.transform_volumes_as_ICV_percent(processed_df, volume_list, file_code)
    
    final_df = processed_df.copy(deep=True)
    
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_04')

    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)
    
    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_04', file_name=file_name, prefix='cleaned/single_file', updated_support_file=new_support_file)    
    
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)
            new_support_file = infoSupportFile.get_subjects_and_multiplevisits(key)
    
    # upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )

save_df(df_to_save=new_support_file, output_path=new_name) 



 ---- ADNI-DIAN_Comparison_Study_Data_Subset_05_23_22_23Oct2025_01.csv


 Dummies in ---- ADNI-DIAN_Comparison_Study_Data_Subset_05_23_22_23Oct2025_01.csv


In [ ]:
final_df

In [8]:
updated_metadata

{'file_code': 'ADNI_DIAN_COMPARISON',
 'level': 'cleaned_04',
 'population': ['ADNI1', 'ADNIGO', 'ADNI2'],
 'source': 'ADNI',
 'cofattori': ['APOE',
  'APOE_4',
  'GENDER/female',
  'GENDER/male',
  'MARRY/divorced',
  'MARRY/married',
  'MARRY/single',
  'MARRY/widowed',
  'ETHNICITY/latino',
  'ETHNICITY/not_latino',
  'RACE/Asian',
  'RACE/Black',
  'RACE/Native_american',
  'RACE/White'],
 'predittori': ['CDRSB',
  'CDRGLOB',
  'MMSE',
  'ICV',
  'Hippocampus',
  'AB42_CSF',
  'PT181_CSF',
  'TTAU_CSF',
  'AB40_CSF',
  'AB4240_CSF',
  'TTAU_AB42_CSF',
  'PT181_AB42_CSF',
  'Apositive',
  'Tpositive',
  'Npositive'],
 'norm_scala': ['CDRSB', 'CDRGLOB', 'MMSE'],
 'norm_intervallo': ['ICV',
  'Hippocampus',
  'AB42_CSF',
  'PT181_CSF',
  'TTAU_CSF',
  'AB40_CSF',
  'AB4240_CSF',
  'TTAU_AB42_CSF',
  'PT181_AB42_CSF'],
 'norm_volume': [],
 'cofattori_metadata': {'APOE_4': [0, 2, 'increasing']},
 'norm_scale_value': {'MMSE': [0, 30, 'inverse'],
  'CDRSB': [0, 18, 'increasing'],
  'CDRGL